# 🧬 OncRAG — NO-LAQA Baseline
### Agentic Chunking · Hybrid BM25+FAISS · NO Query Preprocessor

**What's removed vs the LAQA version:**
- `query_analyzer.py` — not uploaded, not used
- Abbreviation expansion (TNBC, NSCLC etc. passed raw)
- Per-query alpha tuning → fixed `alpha=0.5` for all queries
- Multihop sub-question planning
- Out-of-scope detection

**What's kept identical (fair comparison):**
- Same Mistral-7B-Instruct-v0.2 4-bit NF4 model
- Same BM25 + FAISS hybrid retriever + MMR re-ranking
- Same faithfulness self-correction loop
- Same memory / cache / compressor stack
- Same prompts, same evaluator (S.C.O.P.E.E), same 200 questions

## 📦 Step 1 — Install Dependencies
**Run this first after every runtime restart.**

In [ ]:
# Pinned versions — critical to avoid bitsandbytes compatibility errors
!pip install -q "transformers==4.46.3" "bitsandbytes==0.46.1" "accelerate==0.34.2"
!pip install -q sentence-transformers faiss-cpu
!pip install -q rank-bm25 scikit-learn
!pip install -q pdfminer.six
!pip install -q bert-score   # real BERTScore (roberta-large)
print("✅ All packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 102.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.7 MB/s eta 0:00:00


## ✅ Step 2 — Verify Package Versions

In [ ]:
import bitsandbytes, transformers
from transformers.utils import is_bitsandbytes_available
import torch

print(f"bitsandbytes : {bitsandbytes.__version__}  (need 0.46.1)")
print(f"transformers : {transformers.__version__}  (need 4.46.3)")
print(f"bnb available: {is_bitsandbytes_available()}  (need True)")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f"GPU          : {torch.cuda.get_device_name(0)}  ({total:.1f} GB)")

if not is_bitsandbytes_available():
    print("\n⚠️  bitsandbytes not available — run Step 1 again, then restart runtime")
else:
    print("\n✅ Ready to proceed")

bitsandbytes : 0.46.1  (need 0.46.1)
transformers : 4.46.3  (need 4.46.3)
bnb available: True  (need True)
CUDA         : True
GPU          : Tesla T4  (15.6 GB)

✅ Ready to proceed


## 🖥️ Step 3 — GPU Check & Path Setup

In [ ]:
import torch, gc, os, sys

def gpu_mem(label=""):
    if torch.cuda.is_available():
        used  = torch.cuda.memory_allocated()/1e9
        total = torch.cuda.get_device_properties(0).total_memory/1e9
        tag   = f" [{label}]" if label else ""
        print(f"GPU{tag}: {used:.2f}/{total:.2f} GB  (free: {total-used:.2f} GB)")
    else:
        print("⚠️  No GPU")

def free_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# Clear any stale module cache from previous sessions
for mod in list(sys.modules.keys()):
    if any(m in mod for m in ["chain","evaluator","retriever",
                               "chunker","memory"]):
        del sys.modules[mod]

os.chdir("/content")
sys.path.insert(0, "/content")

# NOTE: different index path from LAQA so both can coexist on same Colab
CHUNKS_JSON = "/content/all_chunks.json"
QA_JSON     = "/content/cleaned_output.json"   # your 200 questions
INDEX_PATH  = "/content/hybrid_index_nolaqa.pkl"

gpu_mem("baseline")
print(f"\nPaths configured:")
print(f"  CHUNKS_JSON : {CHUNKS_JSON}")
print(f"  QA_JSON     : {QA_JSON}")
print(f"  INDEX_PATH  : {INDEX_PATH}")

GPU [baseline]: 0.00/15.64 GB  (free: 15.64 GB)

Paths configured:
  CHUNKS_JSON : /content/all_chunks.json
  QA_JSON     : /content/cleaned_output.json
  INDEX_PATH  : /content/hybrid_index_nolaqa.pkl


## 📁 Step 4 — Verify All Files Are Present

Upload these to `/content` before running:
- `chain.py` ← **no-LAQA version** (downloaded above)
- `chunker.py`, `retriever.py`, `memory.py`, `evaluator.py` ← unchanged
- `cleaned_output.json` ← your 200 QA questions
- `all_chunks.json` ← your PDF chunks (or let Step 5 rebuild from PDFs)
- **DO NOT upload** `query_analyzer.py`

In [ ]:
import glob

print("── Python modules ──────────────────────────────────────")
all_ok = True
# query_analyzer.py intentionally excluded
for f in ["chain.py","chunker.py","retriever.py","memory.py","evaluator.py"]:
    ok   = os.path.exists(f"/content/{f}")
    size = os.path.getsize(f"/content/{f}")/1e3 if ok else 0
    print(f"  {'✅' if ok else '❌ MISSING'}  {f:<22} ({size:.0f} KB)")
    if not ok: all_ok = False

# Confirm LAQA analyzer is absent (expected)
qa_present = os.path.exists("/content/query_analyzer.py")
print(f"  {'❌ FOUND (remove it!)' if qa_present else '✅ absent (correct)'}  query_analyzer.py")
if qa_present: all_ok = False

print("\n── Data files ──────────────────────────────────────────")
for f in ["/content/cleaned_output.json", "/content/all_chunks.json"]:
    ok   = os.path.exists(f)
    size = os.path.getsize(f)/1e6 if ok else 0
    print(f"  {'✅' if ok else '❌ MISSING'}  {os.path.basename(f):<22} ({size:.1f} MB)")
    if not ok: all_ok = False

print("\n── Index files (optional — rebuilt if missing) ─────────")
for f in [INDEX_PATH, INDEX_PATH + ".faiss"]:
    ok   = os.path.exists(f)
    size = os.path.getsize(f)/1e6 if ok else 0
    print(f"  {'✅' if ok else '⚠️  will rebuild'}  {os.path.basename(f):<35} ({size:.1f} MB)")

print("\n── PDFs in /content/data (only needed if rebuilding chunks) ─")
pdfs = sorted(glob.glob("/content/data/*.pdf"))
print(f"  Found: {len(pdfs)} PDF files")

print("\n✅ All required files present" if all_ok else "\n❌ Fix missing files above before continuing")

── Python modules ──────────────────────────────────────
  ❌ MISSING  chain.py               (0 KB)
  ❌ MISSING  chunker.py             (0 KB)
  ✅  retriever.py           (6 KB)
  ❌ MISSING  memory.py              (0 KB)
  ✅  evaluator.py           (29 KB)
  ✅ absent (correct)  query_analyzer.py

── Data files ──────────────────────────────────────────
  ✅  cleaned_output.json    (0.1 MB)
  ✅  all_chunks.json        (19.9 MB)

── Index files (optional — rebuilt if missing) ─────────
  ⚠️  will rebuild  hybrid_index_nolaqa.pkl             (0.0 MB)
  ⚠️  will rebuild  hybrid_index_nolaqa.pkl.faiss       (0.0 MB)

── PDFs in /content/data (only needed if rebuilding chunks) ─
  Found: 0 PDF files

❌ Fix missing files above before continuing


## 📄 Step 5 — Load / Build Chunks
**Skip if `all_chunks.json` is already uploaded** — it loads instantly.

In [ ]:
import json as _j
from chunker import agentic_chunk
from collections import Counter

if os.path.exists(CHUNKS_JSON):
    all_chunks = _j.loads(open(CHUNKS_JSON).read())
    print(f"✅ Loaded cached chunks: {len(all_chunks)} total — skipping chunking")
else:
    pdfs       = sorted(glob.glob("/content/data/*.pdf"))
    all_chunks = []
    print(f"Chunking {len(pdfs)} PDFs (first run ~20-40 min)...")
    for i, pdf in enumerate(pdfs, 1):
        stem  = os.path.splitext(os.path.basename(pdf))[0]
        cache = f"/content/{stem}_chunks.json"
        try:
            chunks = agentic_chunk(pdf, use_agent=True, cache_path=cache)
            all_chunks.extend(chunks)
            print(f"  [{i}/{len(pdfs)}] {os.path.basename(pdf)} → {len(chunks)} chunks")
        except Exception as e:
            print(f"  ⚠️  Skipped: {os.path.basename(pdf)}: {e}")

    with open(CHUNKS_JSON, "w") as f:
        _j.dump(all_chunks, f, indent=2)
    print(f"\n✅ Total: {len(all_chunks)} chunks saved to {CHUNKS_JSON}")

topic_counts = Counter(t for c in all_chunks for t in c.get("topics", ["general"]))
print("Topic distribution:", dict(topic_counts.most_common(8)))

## 🔍 Step 6 — Build Hybrid Index (BM25 + FAISS)
**If `hybrid_index_nolaqa.pkl` is already uploaded, this loads it in seconds.**

In [ ]:
import json as _j
from retriever import HybridRetriever

all_chunks = _j.loads(open(CHUNKS_JSON).read())
retriever  = HybridRetriever(index_path=INDEX_PATH)
retriever.build(all_chunks, force_rebuild=False)

gpu_mem("after FAISS build")
print(f"\n✅ Index ready — {len(all_chunks)} chunks")

# Smoke test — raw query, no abbreviation expansion (this is the no-LAQA difference)
test_q   = "What are the treatment options for locally advanced head and neck cancer?"
test_res = retriever.retrieve_with_mmr(test_q, top_k=5)
print(f"\nRetrieval smoke test ({len(test_res)} results):")
for r in test_res:
    print(f"  [score={r['score_hybrid']:.4f} | {r['source']} p.{r['page']}]  "
          f"{r['text'][:80]}...")

[Retriever] Loading cached index from /content/hybrid_index_nolaqa.pkl


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[Retriever] ✅ Loaded 7973 chunks, FAISS: 7973 vectors
GPU [after FAISS build]: 0.09/15.64 GB  (free: 15.55 GB)

✅ Index ready — 7973 chunks

Retrieval smoke test (5 results):
  [score=0.0313 | 3LFEN-580-version1-2020_soumarova_oncology.pdf p.82]  Anatomical location-based classification 
▪  Lip, oral cavity (C 00, C04) 
▪  Ph...
  [score=0.0191 | The MD Anderson Manual of Medical Oncology 3e.pdf p.395]  Chapter 19  Head and Neck Cancer  393
9
1
R
E
T
P
A
H
C
addition to trials invol...
  [score=0.0211 | 116.pdf p.199]  T A B L E
6.1
Stage Grouping
Stage I
Stage II
Stage III
Stage IV
Chapter 6  Canc...
  [score=0.0246 | The MD Anderson Manual of Medical Oncology 3e.pdf p.381]  19 Head and Neck Cancer
Jennifer L. McQuade  
G. Brandon Gunn  
William N. Willi...
  [score=0.0237 | 22.-Textbook-of-Medical-Oncology-Fourth-Edition-Cavalli-Textbook-of-Medical-Oncology-PDFDrive-.pdf p.136]  132 
  Section II:  Disease-Specific Part
inhibitors, such as hydroxyurea), or b...


## 🤖 Step 7 — Load Mistral-7B (4-bit NF4)
`import chain` triggers 4-bit model load (~5 min, ~5 GB VRAM). Run once per session.

In [ ]:
import chain
print("\nPrompt style check:", "Most questions CAN be answered" in chain.STRICT_RAG_SYSTEM)
print("No-LAQA check — query_analysis always None:",
      chain.OncRAGChainNoLAQA is not None)
gpu_mem("after Mistral 4-bit load")
free_gb = (torch.cuda.get_device_properties(0).total_memory
           - torch.cuda.memory_allocated()) / 1e9 if torch.cuda.is_available() else 0
print(f"\n✅ Free VRAM for eval: {free_gb:.1f} GB")

[Chain-NoLAQA] Loading Mistral-7B-Instruct in 4-bit NF4...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

[Chain-NoLAQA] ✅ mistralai/Mistral-7B-Instruct-v0.2 loaded (4-bit NF4)
[Chain-NoLAQA] GPU: 4.2/15.6 GB  (free: 11.4 GB)

Prompt style check: True
No-LAQA check — query_analysis always None: True
GPU [after Mistral 4-bit load]: 4.23/15.64 GB  (free: 11.41 GB)

✅ Free VRAM for eval: 11.4 GB


## 🔬 Step 8 — Verify Files Are Correctly Loaded
**Run this before every evaluation to avoid using stale cached modules.**

In [ ]:
import evaluator, retriever

checks = {
    "rag_answer has max_loops param":
        "max_loops" in evaluator.rag_answer.__code__.co_varnames,
    "BERTScorer (real BERTScore)":
        "BERTScorer" in str(evaluator.bertscore.__code__.co_names),
    "MMR lambda=0.85":
        0.85 in retriever.HybridRetriever.retrieve_with_mmr.__defaults__,
    "Faithfulness retry in rag_answer":
        "faith_threshold" in evaluator.rag_answer.__code__.co_varnames,
    "Empathy in SCOPE":
        "scope_Emp" in open(evaluator.__file__).read(),
    "No query_analyzer imported in chain":
        "query_analyzer" not in str(chain.__file__ or "") and
        not hasattr(chain, "analyze"),
}

all_good = True
for check, result in checks.items():
    flag = "✅" if result else "❌"
    print(f"  {flag}  {check}")
    if not result: all_good = False

print("\n✅ All checks passed — safe to run evaluation" if all_good
      else "\n❌ Fix failing checks — re-upload files and re-run Steps 3+7+8")

[Eval] ✅ Reusing 4-bit LLM from chain.py
  ✅  rag_answer has max_loops param
  ✅  BERTScorer (real BERTScore)
  ✅  MMR lambda=0.85
  ✅  Faithfulness retry in rag_answer
  ✅  Empathy in SCOPE
  ✅  No query_analyzer imported in chain

✅ All checks passed — safe to run evaluation


## 🧪 Step 9 — Quick Spot Check (Optional but Recommended)
Tests 3 questions before committing to the full 200-question eval.

In [ ]:
import json as _j
from evaluator import rag_answer
from retriever import HybridRetriever

_ret = HybridRetriever(INDEX_PATH)
_ret.build(_j.loads(open(CHUNKS_JSON).read()), force_rebuild=False)

spot_qs = [
    "Which threshold of distant recurrence risk is often used to recommend "
    "systemic adjuvant chemotherapy in early stage breast cancer?",
    "Which imaging modalities are used to evaluate the extent of disease in laryngeal cancer?",
    "What is the most common presenting symptom for head and neck cancer?",
]

print("Spot check (no-LAQA: raw queries, alpha=0.5 fixed):\n")
for q in spot_qs:
    # Note: alpha=0.5 hardcoded, no abbreviation expansion
    chunks_ret = _ret.retrieve(q, top_k=5, alpha=0.5)
    ans, iters = rag_answer(q, chunks_ret)
    nf   = "❌ REFUSED" if "not found" in ans.lower() else "✅ ANSWERED"
    print(f"{nf}  iters={iters}")
    print(f"  Q: {q[:70]}")
    print(f"  A: {ans[:120]}\n")

[Retriever] Loading cached index from /content/hybrid_index_nolaqa.pkl
[Retriever] ✅ Loaded 7973 chunks, FAISS: 7973 vectors
Spot check (no-LAQA: raw queries, alpha=0.5 fixed):



/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ ANSWERED  iters=0
  Q: Which threshold of distant recurrence risk is often used to recommend 
  A: The specific threshold for distant recurrence risk to recommend systemic adjuvant chemotherapy in early stage breast can

✅ ANSWERED  iters=0
  Q: Which imaging modalities are used to evaluate the extent of disease in
  A: Post-contrast CT and MRI are used to evaluate the extent of disease in laryngeal cancer. While post-contrast CT is more 

✅ ANSWERED  iters=0
  Q: What is the most common presenting symptom for head and neck cancer?
  A: Based on the context passages provided, there is no explicit mention of the most common presenting symptom for head and 



## 📊 Step 10 — Run Full S.C.O.P.E.E Evaluation (200 questions)
- Set `MAX_Q = 20` for a quick sanity check first (~25 min)
- Set `MAX_Q = 200` for the full run (~4-6 hrs on Colab T4)
- Results auto-checkpoint — safe to resume if interrupted (see Step 11)

In [ ]:
MAX_Q = 200   # ← set to 20 for quick test first

# Clear old checkpoint for a fresh run
!rm -f /content/eval_results_nolaqa/checkpoint.json
!rm -f /content/eval_results_nolaqa/results_*.json
!rm -f /content/eval_results_nolaqa/report_*.json
!rm -f /content/eval_results_nolaqa/report_*.txt

# Reload evaluator fresh
for mod in list(sys.modules.keys()):
    if "evaluator" in mod:
        del sys.modules[mod]

from evaluator import OncRAGEvaluator

ev = OncRAGEvaluator(
    chunks_json    = CHUNKS_JSON,
    questions_json = QA_JSON,
    output_dir     = "/content/eval_results_nolaqa",
    offline        = False,
    max_questions  = MAX_Q,
)
report = ev.run()

[Eval] ✅ Reusing 4-bit LLM from chain.py
[Eval] Building retrieval index...
[Retriever] Building index over 7973 chunks...
[Retriever] ✅ BM25 built
[Retriever] Encoding with sentence-transformers...


Batches:   0%|          | 0/125 [00:00<?, ?it/s]

[Retriever] ✅ FAISS: 7973 vectors
[Retriever] ✅ Index saved

Total:200  Done:0  Remaining:200  Mode:4-bit Mistral-7B (fast eval)

[1/200] Q001  What are the three main anatomical divisions of the lar...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

[Judge] ✅ Llama3-Med42-8B loaded (separate from the Mistral generator)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[DEBUG RAW] '{"faithfulness": 1.0, "context_relevancy": 1.0, "answer_relevance": 1.0, "S": 5, "C": 5, "O": 5, "P": 5, "E": 5, "Emp": 5}'
  → P@5=0.40  Faith=1.00  ROUGE-L=0.197  SCOPE=5.0/5  Emp=5  iters=0
[2/200] Q002  What are the major goals when treating carcinoma of the...
[DEBUG RAW] '{"faithfulness": 1.0, "context_relevancy": 1.0, "answer_relevance": 1.0, "S": 5, "C": 5, "O": 5, "P": 5, "E": 5, "Emp": 5}\n\nThe predicted answer accurately reflects the reference information, maintaining the major goals of treating carcinoma of the larynx as preserving laryngeal function while ensur'
  → P@5=1.00  Faith=1.00  ROUGE-L=0.164  SCOPE=5.0/5  Emp=5  iters=0
[3/200] Q003  Which threshold of distant recurrence risk is often use...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[DEBUG RAW] '{"faithfulness": 0.0, "context_relevancy": 0.0, "answer_relevance": 0.0, "S": 5, "C": 0, "O": 0, "P": 0, "E": 0, "Emp": 0}'
  → P@5=1.00  Faith=0.00  ROUGE-L=0.138  SCOPE=1.7/5  Emp=1  iters=0 [NOT FOUND]
[4/200] Q004  What do the 'ABCDEs' of early malignant melanoma diagno...
[DEBUG RAW] '{"faithfulness": 1.0, "context_relevancy": 0.0, "answer_relevance": 1.0, "S": 5, "C": 5, "O": 5, "P": 5, "E": 5, "Emp": 5}'
  → P@5=0.00  Faith=1.00  ROUGE-L=0.160  SCOPE=5.0/5  Emp=5  iters=0
[5/200] Q005  What is Xeroderma pigmentosum?...
[DEBUG RAW] '{"faithfulness": 1.0, "context_relevancy": 0.0, "answer_relevance": 1.0, "S": 5, "C": 5, "O": 5, "P": 5, "E": 5, "Emp": 5}\n\nThe predicted answer accurately describes Xeroderma pigmentosum as a genetic disorder affecting DNA repair mechanisms, specifically the nucleotide excision repair system. This '
  → P@5=1.00  Faith=1.00  ROUGE-L=0.184  SCOPE=5.0/5  Emp=5  iters=0
[6/200] Q006  Which imaging modalities are used to evaluate the exten

## 🔄 Step 11 — Resume Interrupted Evaluation
Use this if the eval stopped mid-way (Colab timeout etc.). Picks up from checkpoint.

In [ ]:
import json as _j, os

ckpt = "/content/eval_results_nolaqa/checkpoint.json"
if os.path.exists(ckpt):
    done = _j.loads(open(ckpt).read())
    print(f"✅ Checkpoint: {len(done)} done, {200-len(done)} remaining")
else:
    print("No checkpoint — will start from Q001")

import torch
used = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
if used < 2.0:
    print("⚠️  Model not loaded — run Steps 1, 3, 7 first, then come back here")
else:
    print(f"✅ Model still in GPU memory ({used:.1f} GB) — safe to resume")

# Resume (no checkpoint clearing)
for mod in list(sys.modules.keys()):
    if "evaluator" in mod:
        del sys.modules[mod]

from evaluator import OncRAGEvaluator

ev = OncRAGEvaluator(
    chunks_json    = CHUNKS_JSON,
    questions_json = QA_JSON,
    output_dir     = "/content/eval_results_nolaqa",
    offline        = False,
    max_questions  = 200,
)
report = ev.run()

✅ Checkpoint: 200 done, 0 remaining
✅ Model still in GPU memory (11.1 GB) — safe to resume
[Eval] ✅ Reusing 4-bit LLM from chain.py
[Eval] Building retrieval index...
[Retriever] Loading cached index from eval_index.pkl
[Retriever] ✅ Loaded 7973 chunks, FAISS: 7973 vectors
Resuming: 200 already done.

Total:200  Done:200  Remaining:0  Mode:4-bit Mistral-7B (fast eval)


  ONCOLOGY RAG — COMPLETE EVALUATION REPORT (no-LAQA)
  Hybrid BM25/FAISS + Faithfulness Loop (fast eval mode)
  Questions evaluated : 200
  Not-found answers   : 18  (9%)
  Avg agent iters     : 0.10

  -- Retrieval Quality (k=5) --------------------------------------
  ✅ Precision@5       : 0.7890  (target ≥ 0.75)
  ✅ Recall@5          : 0.9800  (target ≥ 0.85)
  ✅ MRR               : 0.9293  (target ≥ 0.85)
  ✅ NDCG@5            : 0.9306  (target ≥ 0.8)
  ✅ Hit-Rate@5        : 0.9800  (target ≥ 0.9)
  ✅ Avg rerank        : 0.0271  (target ≥ 0.02)

  -- Generation Lexical (citation-stripped) ------------------------
 

In [ ]:
!pip install -q "transformers==4.46.3" "bitsandbytes==0.46.1" "accelerate==0.34.2" huggingface_hub rank_bm25 faiss-cpu sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 65.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
from huggingface_hub import login
login()

In [ ]:
import os
for f in ["evaluator.py", "retriever.py", "all_chunks.json",
          "results_1784621298.json"]:
    path = f"/content/{f}"
    print(f"{'✅' if os.path.exists(path) else '❌ MISSING'}  {f}")

✅  evaluator.py
✅  retriever.py
✅  all_chunks.json
✅  results_1784621298.json


In [ ]:
with open("evaluator.py") as f:
    code = f.read()

code = code.replace(
    'device_map="auto",\n        trust_remote_code=True)',
    'device_map={"": 0}, max_memory={0: "13GiB"},\n        trust_remote_code=True)'
)

with open("evaluator.py", "w") as f:
    f.write(code)

print("✅ Patched")

✅ Patched


In [ ]:
with open("evaluator.py") as f:
    code = f.read()

code = code.replace(
    'device_map={"": 0}, max_memory={0: "13GiB"},',
    'device_map={"": 0},'
)

with open("evaluator.py", "w") as f:
    f.write(code)

print("✅ Removed max_memory")

✅ Removed max_memory


In [ ]:
import sys
for mod in list(sys.modules.keys()):
    if "evaluator" in mod:
        del sys.modules[mod]

In [ ]:
!grep -n "device_map" evaluator.py

60:                     quantization_config=_bnb, device_map="auto",
111:        _JUDGE_MODEL_ID, quantization_config=bnb, device_map={"": 0},


In [ ]:
# ── Step — Re-judge existing no-LAQA answers with Llama3-Med42-8B ────────────
# Run this AFTER: pip installs + huggingface login (same as baseline).
# Needs these files uploaded to /content/ first:
#   - evaluator1.py  (rename/keep as evaluator.py, or import as below)
#   - retriever.py
#   - all_chunks.json
#   - your no-LAQA results json (the file with 'predicted', 'precision_5' etc.)
#
# What it does:
#   1. Rebuilds the retriever index over all_chunks.json (one-time, few min)
#   2. For each saved question, re-retrieves the SAME top-5 chunks
#      (retrieval is deterministic — alpha=0.5 fixed, no randomness)
#   3. Re-judges faithfulness + SCOPE using Llama3-Med42-8B, WITH real context
#   4. Does NOT regenerate answers — reuses your saved Mistral answers as-is

import json, time
from pathlib import Path

from retriever import HybridRetriever
from evaluator import get_combined_judge   # <- if your file is named evaluator.py, use: from evaluator import get_combined_judge

# ── paths — check these match your uploaded filenames ────────────────────────
RESULTS_FILE = "/content/results_1784621298.json"
CHUNKS_FILE  = "/content/all_chunks.json"
INDEX_PATH   = "/content/hybrid_index_nolaqa_rejudge.pkl"   # separate from original, avoids clobbering
OUTPUT_FILE  = "/content/results_nolaqa_med42judge.json"

# ── Step 1 — rebuild the retriever ────────────────────────────────────────────
print("Building retriever index (one-time, few minutes)...")
all_chunks = json.loads(Path(CHUNKS_FILE).read_text())
retriever  = HybridRetriever(index_path=INDEX_PATH)
retriever.build(all_chunks, force_rebuild=False)
print(f"✅ Retriever ready — {len(all_chunks)} chunks\n")

# ── Step 2 — load saved results ───────────────────────────────────────────────
with open(RESULTS_FILE) as f:
    results = json.loads(f.read())

print(f"Re-judging {len(results)} questions with Llama3-Med42-8B...\n")

# ── Step 3 — re-retrieve + re-judge each question ────────────────────────────
t0 = time.time()
for i, row in enumerate(results, 1):
    question = row["question"]
    ref      = row["reference"]
    pred     = row["predicted"]

    # Same retrieval call your notebook used for no-LAQA: alpha=0.5, top_k=5
    chunks = retriever.retrieve(question, top_k=5, alpha=0.5)

    faith, scope = get_combined_judge(question, ref, pred, chunks)

    old_scope = (row["scope_S"] + row["scope_C"] + row["scope_O"] +
                 row["scope_P"] + row["scope_E"] + row["scope_Emp"]) / 6

    row["faithfulness"]      = faith["faithfulness"]
    row["context_relevancy"] = faith["context_relevancy"]
    row["answer_relevance"]  = faith["answer_relevance"]
    row["scope_S"]   = scope["S"]
    row["scope_C"]   = scope["C"]
    row["scope_O"]   = scope["O"]
    row["scope_P"]   = scope["P"]
    row["scope_E"]   = scope["E"]
    row["scope_Emp"] = scope["Emp"]
    row["scope_judge_model"] = "Llama3-Med42-8B"

    new_scope = (scope["S"] + scope["C"] + scope["O"] +
                 scope["P"] + scope["E"] + scope["Emp"]) / 6

    print(f"[{i:>3}/{len(results)}] id={row['id']}  "
          f"SCOPE old={old_scope:.2f} -> new={new_scope:.2f}  "
          f"faith={faith['faithfulness']:.2f}")

    if i % 20 == 0 or i == len(results):
        with open(OUTPUT_FILE, "w") as f:
            json.dump(results, f, indent=2)

elapsed = time.time() - t0
print(f"\n✅ Done in {elapsed/60:.1f} min. Saved -> {OUTPUT_FILE}")

# ── Step 4 — recompute aggregate report ───────────────────────────────────────
from statistics import mean

def avg(key):
    return mean(r[key] for r in results)

scope_keys = ["scope_S", "scope_C", "scope_O", "scope_P", "scope_E", "scope_Emp"]
sc = {k: avg(k) for k in scope_keys}
sv = list(sc.values())
weighted_total = mean(sv)

new_report = {
    "mode": "nolaqa_med42judge",
    "n_questions": len(results),
    "retrieval": {k: avg(k) for k in
        ["precision_5", "recall_5", "mrr", "ndcg_5", "hit_rate_5", "avg_rerank"]},
    "lexical": {k: avg(k) for k in
        ["bleu1", "rouge1", "rougeL", "meteor", "answer_f1"]},
    "semantic": {"bertscore_f1": avg("bertscore_f1")},
    "faithfulness": {
        "faithfulness_llm":  avg("faithfulness"),
        "context_relevancy": avg("context_relevancy"),
        "answer_relevance":  avg("answer_relevance"),
    },
    "scope": {
        "S_safety": sc["scope_S"], "C_completeness": sc["scope_C"],
        "O_originality": sc["scope_O"], "P_precision": sc["scope_P"],
        "E_efficiency": sc["scope_E"], "Emp_empathy": sc["scope_Emp"],
        "weighted_total": weighted_total,
    },
}

print("\n" + "=" * 60)
print("  NO-LAQA — Med42-8B-JUDGED RESULTS")
print("=" * 60)
for key, label in [
    ("S_safety", "S  Safety      "), ("C_completeness", "C  Completeness"),
    ("O_originality", "O  Originality "), ("P_precision", "P  Precision   "),
    ("E_efficiency", "E  Efficiency  "), ("Emp_empathy", "E  Empathy     "),
]:
    print(f"  {label}: {new_report['scope'][key]:.2f}/5.00")
print(f"  Weighted Total    : {new_report['scope']['weighted_total']:.2f}/5.00")
print(f"  Faithfulness      : {new_report['faithfulness']['faithfulness_llm']:.4f}")
print(f"  Context Relevancy : {new_report['faithfulness']['context_relevancy']:.4f}")
print(f"  Answer Relevance  : {new_report['faithfulness']['answer_relevance']:.4f}")
print("=" * 60)

with open("/content/report_nolaqa_med42judge.json", "w") as f:
    json.dump(new_report, f, indent=2)
print("\n✅ Report saved -> /content/report_nolaqa_med42judge.json")

[Eval] Loading LLM independently...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

[Judge] ✅ Llama3-Med42-8B loaded (separate from the Mistral generator)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[DEBUG RAW] '{"faithfulness": 1.0, "context_relevancy": 1.0, "answer_relevance": 1.0, "S": 5, "C": 5, "O": 5, "P": 5, "E": 5, "Emp": 5}'
[  1/200] id=Q001  SCOPE old=5.00 -> new=5.00  faith=1.00
[DEBUG RAW] '{"faithfulness": 1.0, "context_relevancy": 1.0, "answer_relevance": 1.0, "S": 5, "C": 5, "O": 5, "P": 5, "E": 5, "Emp": 5}\n\nThe predicted answer accurately reflects the reference information, maintaining the major goals of treating carcinoma of the larynx as preserving laryngeal function while ensur'
[  2/200] id=Q002  SCOPE old=5.00 -> new=5.00  faith=1.00
[DEBUG RAW] '{"faithfulness": 0.0, "context_relevancy": 0.0, "answer_relevance": 0.0, "S": 5, "C": 0, "O": 0, "P": 0, "E": 0, "Emp": 0}'
[  3/200] id=Q003  SCOPE old=1.67 -> new=1.67  faith=0.00
[DEBUG RAW] '{"faithfulness": 1.0, "context_relevancy": 0.0, "answer_relevance": 1.0, "S": 5, "C": 5, "O": 5, "P": 5, "E": 5, "Emp": 5}'
[  4/200] id=Q004  SCOPE old=5.00 -> new=5.00  faith=1.00
[DEBUG RAW] '{"faithfulness": 1.0, "cont

In [ ]:
import json

with open("results_1784621298.json") as f:
    original = json.load(f)

print("TYPE:", type(original))
if isinstance(original, list):
    print("LENGTH:", len(original))
    print("FIRST ITEM KEYS:", list(original[0].keys()))
    print("FIRST ITEM:", json.dumps(original[0], indent=2)[:800])
else:
    print("KEYS:", list(original.keys()))

print("\n" + "="*40)

with open("report_nolaqa_med42judge.json") as f:
    med42 = json.load(f)

print("MED42 TYPE:", type(med42))
print("MED42 CONTENT:", json.dumps(med42, indent=2)[:1500])

TYPE: <class 'list'>
LENGTH: 200
FIRST ITEM KEYS: ['id', 'category', 'difficulty', 'question', 'reference', 'predicted', 'precision_5', 'recall_5', 'mrr', 'ndcg_5', 'hit_rate_5', 'avg_rerank', 'bleu1', 'bleu2', 'bleu4', 'gleu', 'rouge1', 'rouge2', 'rougeL', 'rougeLsum', 'meteor', 'answer_f1', 'bertscore_f1', 'faithfulness', 'context_relevancy', 'answer_relevance', 'scope_S', 'scope_C', 'scope_O', 'scope_P', 'scope_E', 'scope_Emp', 'agent_iters', 'not_found']
FIRST ITEM: {
  "id": "Q001",
  "category": "diagnosis",
  "difficulty": "simple",
  "question": "What are the three main anatomical divisions of the larynx?",
  "reference": "The larynx is anatomically divided into the supraglottic larynx, the glottis, and the subglottis .",
  "predicted": "The larynx is a complex organ with three main anatomical subsites: the glottis, which contains the true vocal cords; the supraglottis, which is above the level of the vocal cords; and the subglottis, which is below the vocal cords [2].\n\nRefer

In [ ]:
from google.colab import files

files.download("report_nolaqa_med42judge.json")
files.download("results_nolaqa_med42judge.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download(OUTPUT_FILE)                                # results_nolaqa_med42judge.json
files.download("/content/report_nolaqa_med42judge.json")   # aggregate report

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 📈 Step 12 — Results Dashboard

In [ ]:
def dashboard(r, results):
    sep = "=" * 70
    nf  = r.get("not_found_count", "?")
    pct = int(nf/r['n_questions']*100) if isinstance(nf,int) else "?"
    print(f"\n{sep}")
    print("  OncRAG NO-LAQA — EVALUATION DASHBOARD")
    print(f"  {r['n_questions']} questions  |  Not-found: {nf} ({pct}%)  |  "
          f"Avg agent iters: {r['avg_agent_iters']:.2f}")
    print(sep)

    ret = r["retrieval"]
    print("\n  ── RETRIEVAL  (k=5) ─────────────────────────────────────────────")
    for name, val, tgt in [
        ("Precision@5", ret["precision_5"], 0.75),
        ("Recall@5",    ret["recall_5"],    0.85),
        ("MRR",         ret["mrr"],         0.85),
        ("NDCG@5",      ret["ndcg_5"],      0.80),
        ("Hit-Rate@5",  ret["hit_rate_5"],  0.90),
        ("Avg rerank",  ret["avg_rerank"],  0.02),
    ]:
        flag = "✅" if val >= tgt else "⚠️ "
        print(f"  {flag} {name:<18}: {val:.4f}  (target ≥ {tgt})")

    lex = r["lexical"]
    print("\n  ── GENERATION — Lexical (citation-stripped) ─────────────────────")
    for k, v in lex.items():
        print(f"     {k:<20}: {v:.4f}")

    print("\n  ── GENERATION — Semantic ────────────────────────────────────────")
    print(f"     BERTScore F1        : {r['semantic']['bertscore_f1']:.4f}")

    faith = r["faithfulness"]
    print("\n  ── FAITHFULNESS & RELEVANCE ─────────────────────────────────────")
    for k, v in faith.items():
        flag = "✅" if v >= 0.70 else "⚠️ "
        print(f"  {flag} {k:<26}: {v:.4f}")

    sc = r["scope"]
    print("\n  ── S.C.O.P.E.E  LLM-as-judge (/5.0) ───────────────────────────")
    for key, label in [
        ("S_safety",       "S  Safety      "),
        ("C_completeness", "C  Completeness"),
        ("O_originality",  "O  Originality "),
        ("P_precision",    "P  Precision   "),
        ("E_efficiency",   "E  Efficiency  "),
        ("Emp_empathy",    "E  Empathy     "),
    ]:
        v    = sc.get(key, "N/A")
        flag = "✅" if isinstance(v,(int,float)) and v >= 3.5 else "⚠️ "
        print(f"  {flag} {label}: {v}")
    wt   = sc["weighted_total"]
    flag = "✅" if wt >= 3.5 else "⚠️ "
    print(f"  {flag} {'Weighted Total':<18}: {wt:.2f}/5.00  (std={sc['std']:.2f})")
    print(sep)

    if results:
        print("\n  ── Per-Difficulty Breakdown ─────────────────────────────────────")
        from collections import defaultdict
        by_diff = defaultdict(list)
        for row in results: by_diff[row["difficulty"]].append(row)
        for diff in ["simple","moderate","complex"]:
            rows = by_diff.get(diff, [])
            if not rows: continue
            p5   = sum(r["precision_5"] for r in rows)/len(rows)
            sc_a = sum((r["scope_S"]+r["scope_C"]+r["scope_O"]+
                        r["scope_P"]+r["scope_E"]+r.get("scope_Emp",3))/6
                       for r in rows)/len(rows)
            nf_c = sum(1 for r in rows if r.get("not_found",False))
            ai   = sum(r.get("agent_iters",0) for r in rows)/len(rows)
            print(f"     {diff:<10} n={len(rows):<4} P@5={p5:.3f}  "
                  f"SCOPE={sc_a:.1f}/5  not_found={nf_c}  avg_iters={ai:.2f}")

        print("\n  ── Per-Category Breakdown ───────────────────────────────────────")
        by_cat = defaultdict(list)
        for row in results: by_cat[row["category"]].append(row)
        for cat in sorted(by_cat):
            rows = by_cat[cat]
            p5   = sum(r["precision_5"] for r in rows)/len(rows)
            sc_a = sum((r["scope_S"]+r["scope_C"]+r["scope_O"]+
                        r["scope_P"]+r["scope_E"]+r.get("scope_Emp",3))/6
                       for r in rows)/len(rows)
            emp  = sum(r.get("scope_Emp",3) for r in rows)/len(rows)
            print(f"     {cat:<22} n={len(rows):<4} P@5={p5:.3f}  "
                  f"SCOPE={sc_a:.1f}/5  Empathy={emp:.1f}")

import glob as _g, json as _j
result_files = sorted(_g.glob("/content/eval_results_nolaqa/results_*.json"))
if result_files:
    latest = _j.loads(open(result_files[-1]).read())
    dashboard(report, latest)
else:
    print("No results yet — run Step 10 first")

No results yet — run Step 10 first


## 💬 Step 14 — Interactive Chat Demo

In [ ]:
from chain import build_chain

onc_chain = build_chain(
    chunks_json = CHUNKS_JSON,
    index_path  = INDEX_PATH,
    verbose     = True,
)

def chat(question):
    print(f"\n{'='*65}")
    print(f"Q: {question}")
    result = onc_chain.ask(question)
    print(f"\nA: {result['answer']}")
    if result.get("faithfulness") is not None:
        print(f"Faithfulness    : {result['faithfulness']:.2f}")
    print(f"Agent iters     : {result.get('agent_iters', 0)}")
    print(f"Latency         : {result['latency_ms']} ms")
    print(f"Query analysis  : {result['query_analysis']}  ← always None (no LAQA)")
    for s in result["sources"][:3]:
        print(f"  [{s['source']} p.{s['page']} score={s['score']}]")
    gpu_mem()

chat("What are the three main anatomical divisions of the larynx?")
chat("How does HER2 overexpression affect breast cancer prognosis?")
chat("What are the NCCN guidelines for colorectal cancer staging?")

## 💾 Step 15 — Download Results

In [ ]:
import glob as _g
from google.colab import files as _f

result_files = sorted(
    _g.glob("/content/eval_results_nolaqa/*.json") +
    _g.glob("/content/eval_results_nolaqa/*.txt")
)

if result_files:
    print(f"Downloading {len(result_files)} files:")
    for f in result_files:
        size = os.path.getsize(f)/1e3
        print(f"  {os.path.basename(f)}  ({size:.0f} KB)")
        _f.download(f)
else:
    print("No result files yet — run Step 10 first")

  checkpoint.json  (335 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  report_1784621280.json  (1 KB)


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  results_1784621298.json  (335 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>